In [20]:
#libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("Student Performance Analysis and Prediction.csv")
df.head()

,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score,grade
0,1,18.5,95.6,3.8,97.9,A
1,2,14.0,80.0,2.5,83.9,B
2,3,19.5,86.3,5.3,100.0,A
3,4,25.7,70.2,7.0,100.0,A
4,5,13.4,81.9,6.9,92.0,A


In [25]:
df.shape

(1000000, 6)

In [26]:
df.columns

Index(['student_id', 'weekly_self_study_hours', 'attendance_percentage',
       'class_participation', 'total_score', 'grade'],
      dtype='object')

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 6 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   student_id               1000000 non-null  int64  
 1   weekly_self_study_hours  1000000 non-null  float64
 2   attendance_percentage    1000000 non-null  float64
 3   class_participation      1000000 non-null  float64
 4   total_score              1000000 non-null  float64
 5   grade                    1000000 non-null  object 
dtypes: float64(4), int64(1), object(1)
memory usage: 45.8+ MB


In [28]:
df.isnull().sum()

,0
student_id,0
weekly_self_study_hours,0
attendance_percentage,0
class_participation,0
total_score,0
grade,0


In [5]:
df.select_dtypes(exclude='object').corr()

,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score
student_id,1.000000,-0.001607,-0.000353,0.001277,-0.000492
weekly_self_study_hours,-0.001607,1.000000,-0.001008,0.001244,0.812241
attendance_percentage,-0.000353,-0.001008,1.000000,-0.000043,-0.001014
class_participation,0.001277,0.001244,-0.000043,1.000000,0.000684
total_score,-0.000492,0.812241,-0.001014,0.000684,1.000000


We can see that "weekly_self_study_hours" has strong correlatoin with "total_score"





Let's investigate the characteristics of students who achieve a perfect total_score of 100.

In [6]:
df[df['total_score']==100].sort_values(by='weekly_self_study_hours', ascending=False)


,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score,grade
425630,425631,40.0,89.0,8.4,100.0,A
425349,425350,40.0,90.4,2.0,100.0,A
391259,391260,40.0,81.2,8.6,100.0,A
428200,428201,40.0,81.1,5.6,100.0,A
264709,264710,40.0,95.0,7.6,100.0,A
...,...,...,...,...,...,...
24817,24818,6.1,100.0,5.9,100.0,A
83562,83563,5.8,78.0,5.6,100.0,A
18248,18249,5.4,74.2,4.3,100.0,A
626574,626575,5.1,65.4,4.8,100.0,A


Interestingly, a student can achieve a perfect score with up to 40 hours of weekly self-study. This finding underscores the potential for high achievement through dedicated individual effort, even if other factors like class participation or attendance are not always at their peak.

Conversely, let's look at students who achieve a perfect score with minimal self-study.

In [7]:
df[df['total_score']==100].sort_values(by='weekly_self_study_hours').head(5)


,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score,grade
348411,348412,4.6,75.3,5.0,100.0,A
626574,626575,5.1,65.4,4.8,100.0,A
18248,18249,5.4,74.2,4.3,100.0,A
83562,83563,5.8,78.0,5.6,100.0,A
24817,24818,6.1,100.0,5.9,100.0,A


These instances highlight that some students can achieve perfect scores with less than 10 hours of weekly study. This suggests the importance of focused and effective study habits, rather than just raw hours, for certain individuals.



In [8]:
df.describe()


,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,500000.500000,15.029127,84.711046,5.985203,84.283845
std,288675.278933,6.899431,9.424143,1.956421,15.432969
min,1.000000,0.000000,50.000000,0.000000,9.400000
25%,250000.750000,10.300000,78.300000,4.700000,73.900000
50%,500000.500000,15.000000,85.000000,6.000000,87.500000
75%,750000.250000,19.700000,91.800000,7.300000,100.000000
max,1000000.000000,40.000000,100.000000,10.000000,100.000000


Impact of Attendance Percentage
What is the lowest attendance percentage observed among students who achieved a perfect total score?

In [9]:
df[df['total_score']==100]['attendance_percentage'].min()


50.0

A minimum attendance of 50% for a perfect score implies that while self-study is crucial, a baseline level of class attendance is still beneficial. Classes can provide foundational concepts or clarify doubts, complementing individual study.



Let's see the specific cases of students with perfect scores and lower attendance.



In [10]:
df[df['total_score']==100].sort_values(by='attendance_percentage').head(5)


,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score,grade
319359,319360,23.6,50.0,5.7,100.0,A
940067,940068,17.3,50.0,5.3,100.0,A
838754,838755,19.9,50.0,4.9,100.0,A
361282,361283,17.9,50.0,4.0,100.0,A
53889,53890,19.7,50.0,5.2,100.0,A


This further illustrates the diverse paths to academic success. For some students, consistent class participation and attendance might boost confidence and understanding, while for others, intense self-dedication (35-40 hours weekly) is the primary driver for achieving top scores

Preparing Data for Modeling

Moving from exploratory analysis, we now aim to build a predictive model for total_score. For this task, we will focus on weekly_self_study_hours as a key predictor, given its strong correlation with the target variable.

To manage computational resources efficiently for modeling, we'll work with a sampled subset of our dataset. We will also drop the grade column as it is not used in our predictive task for total_score.

In [14]:
df_sampled = df.sample(50000, random_state=42).drop(columns=['grade'])

In [15]:
df_sampled

,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score
987231,987232,3.9,96.7,6.7,53.9
79954,79955,17.1,73.9,5.4,77.3
567130,567131,8.7,98.7,3.0,75.0
500891,500892,15.0,92.7,0.1,85.7
55399,55400,0.9,87.0,5.4,30.8
...,...,...,...,...,...
662925,662926,9.7,88.7,5.3,84.5
649864,649865,24.3,88.5,10.0,100.0
331385,331386,14.7,83.8,8.5,86.4
699967,699968,10.3,71.4,9.3,89.1


In [16]:
df_sampled.corr()

,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score
student_id,1.000000,0.005053,0.003623,-0.000512,0.006324
weekly_self_study_hours,0.005053,1.000000,-0.005812,0.006669,0.811911
attendance_percentage,0.003623,-0.005812,1.000000,-0.002394,-0.009381
class_participation,-0.000512,0.006669,-0.002394,1.000000,0.005094
total_score,0.006324,0.811911,-0.009381,0.005094,1.000000


# Predictive Modeling of Total Score
We will now train several regression models to predict total_score based on weekly_self_study_hours. We'll compare their performance to identify the most effective model.

In [17]:
X = df_sampled[['weekly_self_study_hours']]
y = df_sampled['total_score']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [19]:
# Models to Compare with tuned hyperparameters
models = {
    "DecisionTree": DecisionTreeRegressor(
        max_depth=100,
        random_state=42
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=500,
        max_depth=100,
        random_state=42,
        n_jobs=-1 # Use all CPU cores
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.2,
        max_depth=100,
        subsample=0.8,
        random_state=42
    ),
    "XGBRegressor": XGBRegressor(
        n_estimators=500,
        learning_rate=0.2,
        max_depth=150,
        random_state=42,
        n_jobs=-1 # Use all CPU cores
    )
}

In [21]:
results = []
best_score = -np.inf
best_model = None

In [23]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred)

    results.append({"Model": name, "R2": r2, "MAE": mae, "RMSE": rmse})

    # Select the best model based on R²
    if r2 > best_score:
        best_score = r2
        best_model = model

# Show Results
results_df = pd.DataFrame(results)
print("\nModel Performance Summary:")
print(results_df)

print("\nBest Model Identified:", type(best_model).__name__)


Model Performance Summary:
              Model        R2       MAE       RMSE
0      DecisionTree  0.716174  6.082276  66.977573
1      RandomForest  0.716210  6.081738  66.969059
2  GradientBoosting  0.716236  6.080580  66.962861
3      XGBRegressor  0.716505  6.079341  66.899402

Best Model Identified: XGBRegressor


# Concluding Remarks

This analysis highlights the significant role of dedicated self-study and consistent attendance in student performance. While individual talents and focused study habits can lead to success with fewer hours, sustained effort consistently correlates with higher academic scores. The predictive model further quantifies this relationship, offering a clear insight into how study hours translate into total scores.